# PointTwo · an embedding distance lab

Explore real **Qwen3-Embedding-0.6B** vectors for six synthetic client/matter records and six time-entry narratives.

1. Inspect exactly what text is embedded.
2. Compare **cosine distance in all 1,024 dimensions**.
3. Explore a selectable 2D map and nearest matters.
4. Compare two points and see how much the 2D projection distorts distances.
5. Optionally embed your own narrative locally.

**Run → Run All Cells** to activate the controls. Outputs are cleared for sharing; run the cells to generate plots and tables. This notebook reads application caches in read-only mode; it never edits assignments, feedback, or application embeddings. Missing vectors are generated through local Ollama and saved only in the notebook's private cache.

In [ ]:
%matplotlib inline
from pathlib import Path
import hashlib
import json
import os
import psycopg
from pgvector.psycopg import register_vector
import urllib.request
import urllib.parse
import html
import io
import base64

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "fixtures/catalog.json").exists() and (p / "classifier/engine.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the PointTwo repository or its notebooks directory.")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
pd.set_option("display.max_colwidth", 100)

## Configuration

The default reads only embeddings for the synthetic fixtures, even when Postgres also contains other entries. Set `DATABASE_URL` privately before launching Jupyter to reuse the application's cache. Without it, vectors are generated through local Ollama and cached in the notebook output directory.

No model service is needed when all requested vectors are cached. Nearest-neighbor tables refer to the **displayed sample**, not the full application catalog.


In [ ]:
MODEL = "qwen3-embedding:0.6b"
MODEL_VERSION = "qwen3-0.6b:1024:v1"
DIMENSIONS = 1024
QUERY_PREFIX = "Instruct: Retrieve the client and matter matching this work description.\nQuery: "
MODEL_KEY = None  # Optional complete cached model identity.
OLLAMA_URLS = ["http://127.0.0.1:11434"]
DATABASE_URL = os.environ.get("DATABASE_URL")  # Optional; never print credentials.
OUTPUT = ROOT / ".pointtwo/notebooks"
OUTPUT.mkdir(parents=True, exist_ok=True, mode=0o700)


## The text behind each point

Catalog text follows `classifier/models.py`: client name, matter name, description, keywords, practice area, and matter type. Entries follow `classifier/embeddings.py`: the retrieval instruction plus the **Narrative**.

Imported client/matter assignments can be wrong. They are shown for comparison, but are not embedded as labels. User IDs, billed hours, and dates are also not part of these vectors. Query and document text deliberately use different preparation rules.

In [ ]:
catalog = json.loads((ROOT / "fixtures/catalog.json").read_text())
entries = json.loads((ROOT / "fixtures/time-entries.json").read_text())
records = []
for i, c in enumerate(catalog, 1):
    text = " ".join([c["ClientName"], c["MatterName"], c.get("MatterDescription", ""),
                     " ".join(c.get("RelatedKeywords", [])), c.get("PracticeArea", ""),
                     c.get("MatterType", "")])  # Preserve whitespace: the app hashes this exact string.
    records.append(dict(point=f"C{i}", kind="Catalog", label=f"{c['ClientName']} / {c['MatterName']}",
                        text=text, embedding_input=text, imported_assignment="", status=c.get("Status", "Open")))
for i, e in enumerate(entries, 1):
    records.append(dict(point=f"E{i}", kind="Entry", label=e["EntryId"], text=e["Narrative"],
                        embedding_input=QUERY_PREFIX + e["Narrative"],
                        imported_assignment=f"{e['ClientName']} / {e['MatterName']}", status=""))
display(pd.DataFrame(records)[["point", "kind", "label", "text", "imported_assignment"]])

## Load compatible vectors, reusing the app's cache

The cache key includes the model digest, preprocessing version, and exact embedding input. A cache-only run can explore that saved model version even if the model is not currently running. If a vector is missing, the local model's identity must match before inference is allowed.

Ollama is restricted to literal loopback addresses; proxies, redirects, and cloud model aliases are rejected. There are no API credentials in this notebook.

In [ ]:
def readonly_db():
    connection = psycopg.connect(DATABASE_URL, options="-c default_transaction_read_only=on")
    register_vector(connection)
    return connection

class NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, *args, **kwargs):
        raise RuntimeError("Redirects are disabled for local embedding requests")

http = urllib.request.build_opener(urllib.request.ProxyHandler({}), NoRedirect())
def local_json(origin, route, body=None):
    url = urllib.parse.urlsplit(origin)
    if not (url.scheme == "http" and url.hostname in ("127.0.0.1", "::1")
            and not url.username and not url.password and url.path in ("", "/")
            and not url.query and not url.fragment):
        raise ValueError("Use a literal loopback HTTP origin for Ollama")
    request = urllib.request.Request(origin.rstrip("/") + route,
        data=None if body is None else json.dumps(body).encode(),
        headers={"Content-Type": "application/json"})
    with http.open(request, timeout=180 if route == "/api/embed" else 5) as response:
        return json.load(response)

def local_model_identity():
    if "/" in MODEL or "cloud" in MODEL:
        raise ValueError("Use a local model alias")
    for origin in OLLAMA_URLS:
        try:
            info = next(m for m in local_json(origin, "/api/tags")["models"] if m["name"] == MODEL)
            capabilities = local_json(origin, "/api/show", {"model": MODEL})
            if any(x.get("remote_host") is not None or x.get("remote_model") is not None
                   for x in (info, capabilities)):
                continue
            if "embedding" not in capabilities.get("capabilities", []):
                continue
            return origin, f"{MODEL_VERSION}:{MODEL}:{info['digest']}"
        except (OSError, ValueError, KeyError, StopIteration):
            continue
    raise RuntimeError("Start PointTwo or local Ollama with qwen3-embedding:0.6b installed, then rerun.")

def validate_vectors(vectors):
    array = np.asarray(vectors, dtype=np.float64)
    if (array.ndim != 2 or array.shape[1] != DIMENSIONS or not np.isfinite(array).all()
            or np.any(np.linalg.norm(array, axis=1) == 0)):
        raise ValueError("Expected nonzero, finite 1,024-dimensional vectors")
    return array

def cached_identity():
    if DATABASE_URL:
        with readonly_db() as db:
            row = db.execute("SELECT model_id FROM pt_embedding_cache WHERE model_id LIKE %s ORDER BY model_id LIMIT 1",
                             (f"{MODEL_VERSION}:{MODEL}:%",)).fetchone()
        if row:
            return row[0]
    return None

model_key = MODEL_KEY or cached_identity() or local_model_identity()[1]
lab_cache = OUTPUT / ("vectors-" + hashlib.sha256(model_key.encode()).hexdigest()[:20] + ".npz")

def load_vectors(inputs):
    hashes = [hashlib.sha256(f"{model_key}:{text}".encode()).hexdigest() for text in inputs]
    local_cache = {}
    if lab_cache.exists():
        with np.load(lab_cache, allow_pickle=False) as saved:
            local_cache = {key: saved[key].copy() for key in saved.files}
    values, sources = [None] * len(inputs), [None] * len(inputs)
    if DATABASE_URL:
        with readonly_db() as db:
            for i, key in enumerate(hashes):
                row = db.execute("SELECT embedding FROM pt_embedding_cache WHERE content_hash=%s AND model_id=%s",
                                 (key, model_key)).fetchone()
                if row:
                    values[i] = np.asarray(row[0], dtype=np.float32).copy()
                    sources[i] = "Application cache (read-only)"
    for i, key in enumerate(hashes):
        if values[i] is None and key in local_cache:
            values[i], sources[i] = local_cache[key], "Notebook cache"
    missing = [i for i, value in enumerate(values) if value is None]
    if missing:
        origin, actual_key = local_model_identity()
        if actual_key != model_key:
            raise RuntimeError(f"Model identity changed. Set MODEL_KEY to {actual_key!r} and rerun; do not mix spaces.")
        result = local_json(origin, "/api/embed", {"model": MODEL,
            "input": [inputs[i] for i in missing], "truncate": False, "keep_alive": "30s"})
        generated = validate_vectors(result["embeddings"])
        if len(generated) != len(missing):
            raise ValueError("Embedding response count mismatch")
        for i, vector in zip(missing, generated):
            values[i], sources[i] = vector, "Local Ollama"
            local_cache[hashes[i]] = vector.astype(np.float32)
        np.savez_compressed(lab_cache, **local_cache)
        lab_cache.chmod(0o600)
    return validate_vectors(values), sources

vectors, sources = load_vectors([r["embedding_input"] for r in records])
print("Embedding identity:", model_key)
display(pd.Series(sources).value_counts().rename("Vectors loaded").to_frame())

In [ ]:
points = pd.DataFrame(records)
assert len(points) == len(vectors) and points["point"].is_unique
print(f"Exploring {len(points)} points × {vectors.shape[1]} dimensions.")


## Exact cosine distances

Normalize each vector to unit length. For two normalized vectors \(u,v\):

\[
\text{similarity}=u\cdot v,\qquad d_{\cos}=1-u\cdot v.
\]

**Smaller is closer.** Distance 0 means the same direction, 1 means orthogonal, and 2 means opposite directions. These are geometry measurements, not probabilities or PointTwo's final classification scores. PointTwo also uses lexical evidence, explicit client names, timekeeper context, and feedback.

The matrix below compares entries with matters. It measures the full vectors; no projection is involved.

In [ ]:
unit = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
similarity = np.clip(unit @ unit.T, -1.0, 1.0)
distance = 1.0 - similarity
np.fill_diagonal(distance, 0.0)
assert np.allclose(distance, distance.T, atol=1e-12)
assert np.all((distance >= 0) & (distance <= 2))
entry_indices = np.flatnonzero(points["kind"].eq("Entry"))
catalog_indices = np.flatnonzero(points["kind"].eq("Catalog"))
base_catalog_indices = np.flatnonzero(points["point"].str.startswith("C"))

heat = distance[np.ix_(entry_indices, base_catalog_indices)]
fig, ax = plt.subplots(figsize=(12, 6), layout="constrained")
upper = max(float(heat.max()), 0.5)
im = ax.imshow(heat, cmap="viridis", vmin=0, vmax=upper, aspect="auto")
ax.set_xticks(range(len(base_catalog_indices)), [points.iloc[i].point + "\n" + points.iloc[i].label.replace(" / ", "\n") for i in base_catalog_indices], fontsize=8)
ax.set_yticks(range(len(entry_indices)), [points.iloc[i].point + " · " + points.iloc[i].label for i in entry_indices])
for y in range(heat.shape[0]):
    for x in range(heat.shape[1]):
        ax.text(x, y, f"{heat[y,x]:.3f}", ha="center", va="center",
                color="white" if heat[y,x] < upper * .48 else "black")
ax.set(title="Entry → matter cosine distance · all 1,024 dimensions", xlabel="Catalog document")
fig.colorbar(im, ax=ax, label="Cosine distance · smaller is closer")
fig.savefig(OUTPUT / "cosine-distance-heatmap.png", dpi=170, bbox_inches="tight")
plt.show()
display(points.iloc[base_catalog_indices][["point", "label", "status"]].reset_index(drop=True))

## A 2D map, with exact nearest neighbors

PCA projects the **normalized** vectors onto two axes that retain as much variance as possible. The axes are mathematical components, not named legal concepts. Nearby points on this map can be farther apart in dimensions the map omits.

Markers distinguish entry queries from catalog documents. Lines connect the selected point to its nearest catalog points according to **original cosine distance**, even if the drawing makes another point look closer. The default map is saved below; use the selector after running the notebook to explore other points.

In [ ]:
center = unit.mean(axis=0)
_, singular_values, components = np.linalg.svd(unit - center, full_matrices=False)
if np.sum(singular_values ** 2) == 0:
    raise ValueError("All vectors are identical; PCA has no variance to display.")
variance = singular_values ** 2 / np.sum(singular_values ** 2)
coordinates = (unit - center) @ components[:2].T
print(f"Two axes retain {variance[:2].sum():.1%} of variance in this sample.")

def nearest_matters(index, count=5):
    eligible = catalog_indices[catalog_indices != index]
    closest = eligible[np.argsort(distance[index, eligible], kind="stable")[:count]]
    result = points.iloc[closest][["point", "label", "status"]].copy()
    result["cosine_similarity"] = similarity[index, closest]
    result["cosine_distance"] = distance[index, closest]
    return result.reset_index(drop=True)

def draw_map(index, neighbors=3):
    fig, ax = plt.subplots(figsize=(10, 6), layout="constrained")
    for kind, color, marker in [("Catalog", "#26756a", "s"), ("Entry", "#c06c30", "^")]:
        ids = np.flatnonzero(points["kind"].eq(kind))
        ax.scatter(coordinates[ids,0], coordinates[ids,1], c=color, marker=marker,
                   s=65 if kind == "Entry" else 36, alpha=.8, label=kind, zorder=3)
    for i in range(min(12, len(points))):
        ax.annotate(points.iloc[i].point, coordinates[i], xytext=(5, 6),
                    textcoords="offset points", fontsize=9)
    eligible = catalog_indices[catalog_indices != index]
    neighbors_ids = eligible[np.argsort(distance[index, eligible], kind="stable")[:neighbors]]
    for j in neighbors_ids:
        ax.plot(coordinates[[index,j],0], coordinates[[index,j],1], color="#6e7b82", alpha=.5, lw=1)
    ax.scatter(*coordinates[index], s=230, facecolors="none", edgecolors="#232c3b", lw=2, zorder=4)
    ax.set(title=f"Selected {points.iloc[index].point} · lines follow exact cosine neighbors",
           xlabel=f"PC1 ({variance[0]:.1%} of variance)", ylabel=f"PC2 ({variance[1]:.1%} of variance)")
    ax.set_aspect("equal", adjustable="datalim")
    ax.legend(loc="best");ax.grid(alpha=.15)
    return fig

selected_default = int(entry_indices[0])
figure = draw_map(selected_default)
figure.savefig(OUTPUT / "embedding-map.png", dpi=170, bbox_inches="tight")
plt.show()
display(nearest_matters(selected_default).round(4))

In [ ]:
choices = [(f"{r.point} · {r.label}", i) for i, r in points.iterrows()]
selector = widgets.Dropdown(options=choices, value=selected_default, description="Point:",
                            layout=widgets.Layout(width="95%"))
neighbor_count = widgets.IntSlider(value=3, min=1, max=min(8, len(catalog_indices)-1),
                                   description="Lines:", continuous_update=False)
map_output = widgets.HTML()
def update_map(change=None):
    index = selector.value
    figure = draw_map(index, neighbor_count.value)
    buffer = io.BytesIO();figure.savefig(buffer, format="png", dpi=120, bbox_inches="tight");plt.close(figure)
    encoded = base64.b64encode(buffer.getvalue()).decode()
    detail = points.iloc[[index]][["point", "kind", "text", "imported_assignment"]].to_html(index=False, escape=True)
    neighbors = nearest_matters(index).round(4).to_html(index=False, escape=True)
    map_output.value = f'<div style="overflow-x:auto">{detail}<img alt="Selected embedding and exact neighbors" style="max-width:100%" src="data:image/png;base64,{encoded}">{neighbors}</div>'
selector.observe(update_map, names="value")
neighbor_count.observe(update_map, names="value")
display(widgets.VBox([selector, neighbor_count, map_output]))
update_map()

## Compare any two points

Cosine distance is the application's vector-distance measure. For unit vectors, Euclidean distance is `sqrt(2 × cosine_distance)`. The projected Euclidean distance uses only the two PCA coordinates; compare it with the **full-dimensional Euclidean distance**, not directly with cosine distance.

Use C1 versus C2 to inspect two financing matters for different clients. Try E1 versus each of them, or E3 versus C3 and C6 to investigate a potentially confusing match.

In [ ]:
def compare_points(i, j):
    cosine = float(similarity[i,j])
    original_l2 = float(np.linalg.norm(unit[i] - unit[j]))
    projected_l2 = float(np.linalg.norm(coordinates[i] - coordinates[j]))
    assert np.isclose(original_l2, np.sqrt(2 * distance[i,j]), atol=1e-7)
    assert projected_l2 <= original_l2 + 1e-7
    return pd.DataFrame({"Measure": ["Cosine similarity (1,024D)", "Cosine distance (1,024D)",
        "Angle (degrees)", "Euclidean distance (normalized 1,024D)", "Euclidean distance (2D projection)"],
        "Value": [cosine, distance[i,j], np.degrees(np.arccos(cosine)), original_l2, projected_l2]})

display(compare_points(0, 1).round(4))
left = widgets.Dropdown(options=choices, value=0, description="A:", layout=widgets.Layout(width="95%"))
right = widgets.Dropdown(options=choices, value=1, description="B:", layout=widgets.Layout(width="95%"))
pair_output = widgets.HTML()
def update_pair(change=None):
    detail = points.iloc[[left.value, right.value]][["point", "kind", "text"]].to_html(index=False, escape=True)
    comparison = compare_points(left.value, right.value).round(4).to_html(index=False, escape=True)
    pair_output.value = f'<div style="overflow-x:auto">{detail}{comparison}</div>'
left.observe(update_pair, names="value");right.observe(update_pair, names="value")
display(widgets.VBox([left, right, pair_output]));update_pair()

## How much distance does the projection lose?

Each dot below is one pair of points. The dashed diagonal is perfect distance preservation. PCA drops dimensions, so projected Euclidean distances lie on or below that line. A pair near the bottom can look close on the map despite being separated in the original space.

Use the original cosine-distance tables for matching decisions. The map is an explanatory view, not a replacement search index.

In [ ]:
a, b = np.triu_indices(len(points), k=1)
original = np.sqrt(2 * distance[a,b])
projected = np.linalg.norm(coordinates[a] - coordinates[b], axis=1)
assert np.all(projected <= original + 1e-7)
fig, ax = plt.subplots(figsize=(7, 5), layout="constrained")
ax.scatter(original, projected, s=15, alpha=.5, c="#26756a")
maximum = float(original.max()) * 1.05
ax.plot([0, maximum], [0, maximum], "--", color="#9b6646", label="Perfect preservation")
ax.set(xlim=(0, maximum), ylim=(0, maximum), xlabel="Euclidean distance · normalized 1,024D",
       ylabel="Euclidean distance · 2D projection", title="Projection distortion across all displayed pairs")
ax.set_aspect("equal");ax.legend();ax.grid(alpha=.15)
fig.savefig(OUTPUT / "projection-distortion.png", dpi=170, bbox_inches="tight")
plt.show()
# Inspect or export the full small-sample matrix without confusing it with 2D geometry.
cosine_distance_table = pd.DataFrame(distance, index=points.point, columns=points.point)
cosine_distance_table.to_csv(OUTPUT / "cosine-distances.csv")
points[["point", "kind", "label", "text", "imported_assignment", "status"]].to_csv(OUTPUT / "points.csv", index=False)
print(f"Saved a {len(points)} × {len(points)} exact cosine-distance table and three plots under .pointtwo/notebooks/.")

## Try a different narrative (optional)

Edit the text below and run this cell. A missing vector is generated only through local Ollama; the app database stays read-only. The new point is projected into the existing PCA axes **without refitting them**, and compared with the displayed catalog sample.

If you replace synthetic text with real work, notebook outputs will contain that text. Clear outputs before sharing the notebook. Feedback changes PointTwo's ranking and reviewed-example retrieval; it does not move existing vectors in this map unless texts or the embedding model change.

In [ ]:
CUSTOM_NARRATIVE = ""  # Example: "Revised Acme Ventures employee onboarding agreements."
if CUSTOM_NARRATIVE.strip():
    custom_raw, custom_source = load_vectors([QUERY_PREFIX + CUSTOM_NARRATIVE])
    custom_unit = custom_raw[0] / np.linalg.norm(custom_raw[0])
    custom_similarity = np.clip(unit[catalog_indices] @ custom_unit, -1, 1)
    order = np.argsort(-custom_similarity, kind="stable")[:5]
    result = points.iloc[catalog_indices[order]][["point", "label", "status"]].copy()
    result["cosine_similarity"] = custom_similarity[order]
    result["cosine_distance"] = 1 - custom_similarity[order]
    display(result.reset_index(drop=True).round(4))
    custom_xy = (custom_unit - center) @ components[:2].T
    fig = draw_map(selected_default, neighbors=0)
    ax = fig.axes[0]
    ax.scatter(*custom_xy, s=180, c="#9b3e73", marker="*", label="New narrative", zorder=5)
    ax.annotate("New narrative", custom_xy, xytext=(8, 8), textcoords="offset points")
    ax.set_title("New narrative projected into the existing axes");ax.legend()
    plt.show()
    print("Vector source:", custom_source[0])
else:
    print("Set CUSTOM_NARRATIVE above to explore another local embedding.")